In [1]:
import pandas as pd
df = pd.read_csv("Loan.csv")
print(df.head())

  ApplicationDate  Age  AnnualIncome  CreditScore EmploymentStatus  \
0      01-01-2018   45         39948          617         Employed   
1      02-01-2018   38         39709          628         Employed   
2      03-01-2018   47         40724          570         Employed   
3      04-01-2018   58         69084          545         Employed   
4      05-01-2018   37        103264          594         Employed   

  EducationLevel  Experience  LoanAmount  LoanDuration MaritalStatus  ...  \
0         Master          22       13152            48       Married  ...   
1      Associate          15       26045            48        Single  ...   
2       Bachelor          26       17627            36       Married  ...   
3    High School          34       37898            96        Single  ...   
4      Associate          17        9184            36       Married  ...   

   MonthlyIncome UtilityBillsPaymentHistory  JobTenure  NetWorth  \
0    3329.000000                   0.724972     

Data Cleaning

In [2]:
print("Null values: ")
print(df.isnull().sum())

Null values: 
ApplicationDate               0
Age                           0
AnnualIncome                  0
CreditScore                   0
EmploymentStatus              0
EducationLevel                0
Experience                    0
LoanAmount                    0
LoanDuration                  0
MaritalStatus                 0
NumberOfDependents            0
HomeOwnershipStatus           0
MonthlyDebtPayments           0
CreditCardUtilizationRate     0
NumberOfOpenCreditLines       0
NumberOfCreditInquiries       0
DebtToIncomeRatio             0
BankruptcyHistory             0
LoanPurpose                   0
PreviousLoanDefaults          0
PaymentHistory                0
LengthOfCreditHistory         0
SavingsAccountBalance         0
CheckingAccountBalance        0
TotalAssets                   0
TotalLiabilities              0
MonthlyIncome                 0
UtilityBillsPaymentHistory    0
JobTenure                     0
NetWorth                      0
BaseInterestRate          

In [3]:
print("Duplicated Values: ")
print(df.duplicated().sum())

Duplicated Values: 
0


In [4]:
df.columns

Index(['ApplicationDate', 'Age', 'AnnualIncome', 'CreditScore',
       'EmploymentStatus', 'EducationLevel', 'Experience', 'LoanAmount',
       'LoanDuration', 'MaritalStatus', 'NumberOfDependents',
       'HomeOwnershipStatus', 'MonthlyDebtPayments',
       'CreditCardUtilizationRate', 'NumberOfOpenCreditLines',
       'NumberOfCreditInquiries', 'DebtToIncomeRatio', 'BankruptcyHistory',
       'LoanPurpose', 'PreviousLoanDefaults', 'PaymentHistory',
       'LengthOfCreditHistory', 'SavingsAccountBalance',
       'CheckingAccountBalance', 'TotalAssets', 'TotalLiabilities',
       'MonthlyIncome', 'UtilityBillsPaymentHistory', 'JobTenure', 'NetWorth',
       'BaseInterestRate', 'InterestRate', 'MonthlyLoanPayment',
       'TotalDebtToIncomeRatio', 'LoanApproved', 'RiskScore'],
      dtype='object')

In [5]:
loan_factors = df[[
    "Age",
    "AnnualIncome",
    "CreditScore",
    "EmploymentStatus",
    "LoanAmount",
    "LoanDuration",
    "MonthlyDebtPayments",
    "TotalAssets",
    "TotalLiabilities",
    "LoanApproved"    
]].copy()

EDA

In [6]:
print(loan_factors.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Age                  20000 non-null  int64 
 1   AnnualIncome         20000 non-null  int64 
 2   CreditScore          20000 non-null  int64 
 3   EmploymentStatus     20000 non-null  object
 4   LoanAmount           20000 non-null  int64 
 5   LoanDuration         20000 non-null  int64 
 6   MonthlyDebtPayments  20000 non-null  int64 
 7   TotalAssets          20000 non-null  int64 
 8   TotalLiabilities     20000 non-null  int64 
 9   LoanApproved         20000 non-null  int64 
dtypes: int64(9), object(1)
memory usage: 1.5+ MB
None


In [7]:
print(loan_factors["EmploymentStatus"].unique())

['Employed' 'Self-Employed' 'Unemployed']


In [8]:
#Manually mapping EmplymentStatus to numerical values
loan_factors["EmploymentStatus"]=loan_factors["EmploymentStatus"].map({
    "Employed": 0,
    "Self-Employed": 1,
    "Unemployed": 2
})

In [9]:
print(loan_factors.head())

   Age  AnnualIncome  CreditScore  EmploymentStatus  LoanAmount  LoanDuration  \
0   45         39948          617                 0       13152            48   
1   38         39709          628                 0       26045            48   
2   47         40724          570                 0       17627            36   
3   58         69084          545                 0       37898            96   
4   37        103264          594                 0        9184            36   

   MonthlyDebtPayments  TotalAssets  TotalLiabilities  LoanApproved  
0                  183       146111             19183             0  
1                  496        53204              9595             0  
2                  902        25176            128874             0  
3                  755       104822              5370             0  
4                  274       244305             17286             1  


In [10]:
#Calculating the collateral available from assets and liabilites
loan_factors["Collateral"] = (
    loan_factors["TotalAssets"] - loan_factors["TotalLiabilities"]
)

loan_factors.drop(
    columns=["TotalAssets","TotalLiabilities"],
    inplace=True
)

In [11]:
#Calculating debt-to-income ratio
loan_factors["DebttoIncome"] = (
    loan_factors["MonthlyDebtPayments"]/(loan_factors["AnnualIncome"]/12)
)

loan_factors.drop(
    columns=["MonthlyDebtPayments","LoanDuration"],
    inplace=True
)

In [12]:
print(loan_factors.head())

   Age  AnnualIncome  CreditScore  EmploymentStatus  LoanAmount  LoanApproved  \
0   45         39948          617                 0       13152             0   
1   38         39709          628                 0       26045             0   
2   47         40724          570                 0       17627             0   
3   58         69084          545                 0       37898             0   
4   37        103264          594                 0        9184             1   

   Collateral  DebttoIncome  
0      126928      0.054971  
1       43609      0.149890  
2     -103698      0.265789  
3       99452      0.131145  
4      227019      0.031841  


In [13]:
loan_factors.to_csv("Final_data.csv",index=False)

Dataset for shap which contains only features

In [26]:
print(loan_factors.columns)

Index(['Age', 'AnnualIncome', 'CreditScore', 'EmploymentStatus', 'LoanAmount',
       'LoanApproved', 'Collateral', 'DebttoIncome'],
      dtype='object')


In [29]:
Features = loan_factors[[
    'Age', 'AnnualIncome', 'CreditScore', 'EmploymentStatus', 'LoanAmount',
        'Collateral', 'DebttoIncome'
]].copy()

print(Features.columns)

Index(['Age', 'AnnualIncome', 'CreditScore', 'EmploymentStatus', 'LoanAmount',
       'Collateral', 'DebttoIncome'],
      dtype='object')


In [30]:
Features.to_csv("Features.csv",index=False)

Model

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [16]:
X = loan_factors[["Age","AnnualIncome","CreditScore","EmploymentStatus","LoanAmount","Collateral","DebttoIncome"]]
Y = loan_factors["LoanApproved"]

In [17]:
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size=0.2, random_state=42)

rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_train,Y_train)

RandomForestClassifier(random_state=42)

In [18]:
Y_pred = rf_classifier.predict(X_test)

In [19]:
accuracy = accuracy_score(Y_test,Y_pred)
print(f"Accuracy Score: {accuracy*100}")

Accuracy Score: 88.9


XGBoost

In [20]:
import xgboost as xgb
import numpy as np

In [21]:
xgb_train = xgb.DMatrix(X_train, Y_train, enable_categorical=True)
xgb_test = xgb.DMatrix(X_test, Y_test, enable_categorical=True)

In [22]:
params = {
    'objective': 'binary:logistic',
    'max_depth': 7,
    'learning_rate': 0.08,
}
n=50
xg_boost_model = xgb.train(params=params,dtrain=xgb_train,num_boost_round=n)

In [23]:
preds = xg_boost_model.predict(xgb_test)
preds = np.round(preds)

In [24]:
accuracy= accuracy_score(Y_test,preds)
print('Accuracy of the model is:', accuracy*100)

Accuracy of the model is: 89.2


Saving the trained model


In [25]:
xg_boost_model.save_model("Prediction_model.json")